In [ ]:
import spikeinterface as si 
import spikeinterface.preprocessing as spre
from dataLoading_utils import getXMLData
from dataAnalysis_utils import find_repetitive_channels
import numpy as np
import spikeinterface.widgets as sw
import matplotlib.pyplot as plt
from probeinterface import Probe, ProbeGroup, generate_linear_probe, generate_multi_shank, combine_probes 
from probeinterface.plotting import plot_probe
import matplotlib
from pathlib import Path
import pandas as pd
import spikeinterface.extractors as se

dat_path = r"/Volumes/Trenholm2/neuroTechData/Viktor_Budapest/3170_day8_260415_170145/amplifier.dat" 
xml_path = r"/Volumes/Trenholm2/neuroTechData/Viktor_Budapest/3170_day8_260415_170145/amplifier.xml" 

channel_ids, skippedChannels, xml_data = getXMLData(xml_path)
nBits, nChannels, samplingRate, offset, lfpSamplingRate = xml_data

samplingRate = 20000 # for viktor mouse XML is wrong
uV_per_count = 0.195 #https://intantech.com/files/Intan_RHX_user_guide.pdf

full_recording = si.read_binary(file_paths=dat_path, sampling_frequency=samplingRate, dtype=f"int{nBits}", num_channels=nChannels, channel_ids=np.arange(nChannels), is_filtered=False, gain_to_uV=uV_per_count, offset_to_uV=0.0)

shank2_channel_ids = np.array(channel_ids[32:])
shank2_recording = full_recording.select_channels(channel_ids=shank2_channel_ids)

shorter_rec_duration = 30 # minutes
RECORDING = shank2_recording.frame_slice(start_frame=0, end_frame=int(shorter_rec_duration * 60 * samplingRate))

del full_recording
del shank2_recording


########### PROBE 2: MATLAB script ###########

siteLoc = np.array([(37.75, 732), ( 0, 698), ( 36.53225807, 678), ( 1.21774194, 645), ( 35.31451613, 625), ( 2.43548387, 593), ( 34.0967742, 573), ( 3.65322581, 542), ( 32.87903226, 522), ( 4.87096774, 491), ( 31.66129033, 472), ( 6.08870968, 442), ( 30.44354839, 422), ( 7.30645162, 393), ( 29.22580645, 374), ( 8.524193551, 346), ( 28.008064519, 326), ( 9.741935487, 299), ( 26.790322584, 280), ( 10.959677422, 253), ( 25.572580648, 234), ( 12.177419358, 209), ( 24.354838713, 189), ( 13.395161293, 165), ( 23.137096777, 145), ( 14.612903229, 122), ( 21.919354842, 102), ( 15.830645164, 80.22594789), ( 20.701612906, 60.72594789), ( 17.0483871, 39.56206844), ( 19.483870971, 20.06206844), ( 18.266129035, 0), ( 3037.75, 732), ( 3000, 698), ( 3036.53225807, 678), ( 3001.21774194, 645), ( 3035.31451613, 625), ( 3002.43548387, 593), ( 3034.0967742, 573), ( 3003.65322581, 542), ( 3032.87903226, 522), ( 3004.87096774, 491), ( 3031.66129033, 472), ( 3006.08870968, 442), ( 3030.44354839, 422), ( 3007.30645162, 393), ( 3029.22580645, 374), ( 3008.524193551, 346), ( 3028.008064519, 326), ( 3009.741935487, 299), ( 3026.790322584, 280), ( 3010.959677422, 253), ( 3025.572580648, 234), ( 3012.177419358, 209), ( 3024.354838713, 189), ( 3013.395161293, 165), ( 3023.137096777, 145), ( 3014.612903229, 122), ( 3021.919354842, 102), ( 3015.830645164, 80.22594789), ( 3020.701612906, 60.72594789), ( 3017.0483871, 39.56206844), ( 3019.483870971, 20.06206844), ( 3018.266129035, 0)])

probe_m_x = siteLoc[32:,0] - np.min(siteLoc[32:,0]) # shift x coordinates so the minimum is at 0
probe_m_y = siteLoc[32:,1] - np.min(siteLoc[32:,1]) # shift y coordinates so the minimum is at 0

probe_m = Probe(ndim=2, si_units='um')
probe_m.set_contacts(positions=np.column_stack((probe_m_x, probe_m_y)), shapes=np.array(['square'] * 32), shape_params={"width": 13})
# set the contact IDs for each site 
probe_m.set_device_channel_indices(np.arange(32))
probe_m.set_contact_ids(shank2_channel_ids)
probe_m.create_auto_shape()

shank2_probe_m = RECORDING.set_probe(probe_m, in_place=False)

########### PROBE 3: LINEAR ###########

probe_l_x = np.zeros(32)
probe_l_y = np.arange(32) * 19.5

probe_l = Probe(ndim=2, si_units='um')
probe_l.set_contacts(positions=np.column_stack((probe_l_x, probe_l_y)), shapes=np.array(['square'] * 32), shape_params={"width": 13})
# set the contact IDs for each site 
probe_l.set_device_channel_indices(np.arange(32))
probe_l.set_contact_ids(shank2_channel_ids)
probe_l.create_auto_shape()

shank2_probe_l = RECORDING.set_probe(probe_l, in_place=False)

shank2_skipped_channels = np.intersect1d(shank2_channel_ids, skippedChannels)

m_rec_clean = shank2_probe_m.remove_channels(shank2_skipped_channels)
l_rec_clean = shank2_probe_l.remove_channels(shank2_skipped_channels)
clean_channel_ids = m_rec_clean.get_channel_ids()


prePros_m = {}
prePros_l = {}

for rec, out in zip([m_rec_clean, l_rec_clean], [prePros_m, prePros_l]):
    P1 = si.preprocessing.center(rec, mode='median', dtype='float32')
    P2 = si.preprocessing.highpass_filter(P1, freq_min=300.0) # TODO: consider bandpass 
    #P3_glob = si.preprocessing.common_reference(recording = P2, reference = 'global', operator='median') 
    P3_loc = si.preprocessing.common_reference(recording = P2, reference='local', operator='median', local_radius=(40, 180), min_local_neighbors=5)
    #P4_z = si.preprocessing.zscore(P3_loc, mode="median+mad")
    P4_loc = si.preprocessing.whiten(P3_loc, mode="local", radius_um=100) 

    out.update({"P1":P1, "P2":P2, "P3_loc":P3_loc, "P4_loc": P4_loc})

sorting_folders = {
    "simple": Path("/path/to/simple_output"),
    "mountainsort5": Path("/path/to/mountainsort5_output"),
    "spykingcircus2": Path("/path/to/spykingcircus2_output"),
    "kilosort4": Path("/path/to/kilosort4_output"),
}

In [ ]:
recording_for_analysis_m = prePros_m["P3_loc"]
recording_for_analysis_l = prePros_l["P3_loc"] 